# Layer normalisation
Batch norm and layer norm by hand on the same small table, checked against Keras; why padding spoils batch norm on
sentences (a toy batch and a real IMDB batch); and layer norm in the transformer setting.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import numpy as np
import pandas as pd
import keras
import tensorflow as tf

np.set_printoptions(precision=3, suppress=True)
EPS = 0.001                                    # Keras' default epsilon for both layers

## 1. The same table, normalised two ways
5 observations (rows of the batch) and the weighted sums z1, z2, z3 of 3 nodes (columns).

In [2]:
Z = np.array([[7, 5, 4],
              [2, 3, 4],
              [6, 5, 1],
              [3, 5, 4],
              [2, 4, 6]], dtype="float32")
bn_mu, bn_var = Z.mean(0), Z.var(0)            # one mean and variance per column (node), over the batch
ln_mu, ln_var = Z.mean(1, keepdims=True), Z.var(1, keepdims=True)   # one per row (observation), over the nodes
print("batch norm: column means", bn_mu, " column std", np.sqrt(bn_var))
print("layer norm: row means", ln_mu.ravel(), " row std", np.sqrt(ln_var).ravel())
bn_hand = (Z - bn_mu) / np.sqrt(bn_var + EPS)
ln_hand = (Z - ln_mu) / np.sqrt(ln_var + EPS)
print("batch norm by hand:\n", bn_hand)
print("layer norm by hand:\n", ln_hand)
print("layer norm, first row, without epsilon:", ((Z[0] - Z[0].mean()) / Z[0].std()).round(3))

batch norm: column means [4.  4.4 3.8]  column std [2.098 0.8   1.6  ]
layer norm: row means [5.333 3.    4.    4.    4.   ]  row std [1.247 0.816 2.16  0.816 1.633]
batch norm by hand:
 [[ 1.43   0.749  0.125]
 [-0.953 -1.749  0.125]
 [ 0.953  0.749 -1.75 ]
 [-0.477  0.749  0.125]
 [-0.953 -0.5    1.375]]
layer norm by hand:
 [[ 1.336 -0.267 -1.069]
 [-1.224  0.     1.224]
 [ 0.926  0.463 -1.389]
 [-1.224  1.224  0.   ]
 [-1.225  0.     1.225]]
layer norm, first row, without epsilon: [ 1.336 -0.267 -1.069]


Keras' layers (gamma = 1, beta = 0 at the start) give the same numbers. BatchNormalization in training mode uses the
batch statistics.

In [3]:
bn = keras.layers.BatchNormalization()
ln = keras.layers.LayerNormalization()
bn_keras = bn(Z, training=True).numpy()
ln_keras = ln(Z).numpy()
print("batch norm, max |hand - Keras|:", np.abs(bn_hand - bn_keras).max())
print("layer norm, max |hand - Keras|:", np.abs(ln_hand - ln_keras).max())
print("after batch norm: column means", bn_keras.mean(0), " column std", bn_keras.std(0))
print("after layer norm: row means", ln_keras.mean(1), " row std", ln_keras.std(1))

batch norm, max |hand - Keras|: 2.3841858e-07
layer norm, max |hand - Keras|: 2.9802322e-07
after batch norm: column means [-0. -0.  0.]  column std [1.    0.999 1.   ]
after layer norm: row means [ 0.  0.  0.  0. -0.]  row std [1.    0.999 1.    0.999 1.   ]


gamma and beta: one pair per feature (column) in both layers.

In [4]:
for name, layer in (("BatchNormalization", bn), ("LayerNormalization", ln)):
    print(name, [(w.path.split("/")[-1], tuple(w.shape), w.trainable) for w in layer.weights])

BatchNormalization [('gamma', (3,), True), ('beta', (3,), True), ('moving_mean', (3,), False), ('moving_variance', (3,), False)]
LayerNormalization [('gamma', (3,), True), ('beta', (3,), True)]


## 2. Two padded sentences, 3-dimensional vectors
"hi rahul" (2 words) and "how are you today" (4 words) padded to 4 words; padding vectors are zeros.

In [5]:
rng = np.random.default_rng(0)
vocab = ["hi", "rahul", "how", "are", "you", "today"]
vec = {w: rng.uniform(1, 9, size=3).round(1) for w in vocab}
sents = [["hi", "rahul"], ["how", "are", "you", "today"]]
X = np.zeros((2, 4, 3), dtype="float32")
for b, s in enumerate(sents):
    for t, w in enumerate(s):
        X[b, t] = vec[w]
print(X)
rows = X.reshape(-1, 3)                                      # stack the 8 word vectors
real = rows.any(1)
print("batch norm column means with padding:   ", rows.mean(0))
print("batch norm column means, real words only:", rows[real].mean(0))
print("batch norm column std with padding:   ", rows.std(0))
print("batch norm column std, real words only:", rows[real].std(0))

[[[6.1 3.2 1.3]
  [1.1 7.5 8.3]
  [0.  0.  0. ]
  [0.  0.  0. ]]

 [[5.9 6.8 5.3]
  [8.5 7.5 1. ]
  [7.9 1.3 6.8]
  [2.4 7.9 5.3]]]
batch norm column means with padding:    [3.987 4.275 3.5  ]
batch norm column means, real words only: [5.317 5.7   4.667]
batch norm column std with padding:    [3.287 3.295 3.082]
batch norm column std, real words only: [2.709 2.521 2.687]


Keras' BatchNormalization on the (2, 4, 3) tensor uses all 8 rows, padding included: the same as by hand.

In [6]:
bn_seq = keras.layers.BatchNormalization()(X, training=True).numpy().reshape(-1, 3)
print("max |Keras - by hand with padding|:", np.abs(bn_seq - (rows - rows.mean(0)) / np.sqrt(rows.var(0) + EPS)).max())

max |Keras - by hand with padding|: 1.1920929e-07


Layer normalisation of each word vector: the padding rows do not enter any real word's statistics.
A padding row has mean 0 and variance 0, so it becomes (0 - 0)/sqrt(0 + eps) = 0, then gamma * 0 + beta = beta.

In [7]:
ln3 = keras.layers.LayerNormalization()
ln3.build((None, 4, 3))
ln3.beta.assign([0.1, 0.2, 0.3])                             # a non-zero beta, to see where it appears
out = ln3(X).numpy()
print(out)
hi = X[0, 0]
print('hi: mean', hi.mean().round(2), ' std', hi.std().round(2), ' before gamma and beta:', ((hi - hi.mean()) / np.sqrt(hi.var() + EPS)).round(2))

[[[ 1.4    0.031 -0.831]
  [-1.307  0.779  1.128]
  [ 0.1    0.2    0.3  ]
  [ 0.1    0.2    0.3  ]]

 [[-0.062  1.496 -0.834]
  [ 0.952  0.751 -1.103]
  [ 0.989 -1.197  0.808]
  [-1.146  1.402  0.345]]]
hi: mean 3.53  std 1.97  before gamma and beta: [ 1.3  -0.17 -1.13]


Self-attention without a mask does not keep a padding position at zero: its query is 0, so all its scores are 0,
its weights are equal, and its output is the average of the value vectors.

In [8]:
Wq, Wk, Wv = (rng.normal(size=(3, 3)) for _ in range(3))
x = X[0]                                                     # "hi rahul" + 2 padding rows
q, k, v = x @ Wq, x @ Wk, x @ Wv
s = q @ k.T
w = np.exp(s - s.max(1, keepdims=True)); w /= w.sum(1, keepdims=True)
print("weights of the padding row:", w[2].round(3))
print("output of the padding row:", (w @ v)[2].round(3), " mean of the value vectors:", v.mean(0).round(3))

weights of the padding row: [0.25 0.25 0.25 0.25]
output of the padding row: [ 6.048 -1.726  6.096]  mean of the value vectors: [ 6.048 -1.726  6.096]


## 3. A real batch: 32 IMDB reviews

In [9]:
(x_train, _), _ = keras.datasets.imdb.load_data(num_words=10000)
batch = x_train[:32]
lengths = np.array([len(r) for r in batch])
longest = lengths.max()
print("review lengths: min", lengths.min(), " median", int(np.median(lengths)), " max", longest)
print("share of padding when padded to the longest review:", round(1 - lengths.sum() / (32 * longest), 3))

review lengths: min 43  median 175  max 888
share of padding when padded to the longest review: 0.716


Each word becomes an 8-number vector. A random embedding with mean 2 and standard deviation 1 stands in for the
vectors produced by the layer before the normalisation (like the toy numbers above, they are not centred at 0).
Padding positions are set to zero vectors.

In [10]:
keras.utils.set_random_seed(0)
embed = keras.layers.Embedding(10000, 8, embeddings_initializer=keras.initializers.RandomNormal(mean=2.0, stddev=1.0))


def padded_vectors(T):
    ids = keras.utils.pad_sequences(batch, maxlen=T, padding="post")
    v = embed(ids).numpy()
    v[ids == 0] = 0.0
    return v, ids != 0


v, mask = padded_vectors(longest)
real_rows = v[mask]
print("feature 0: batch-norm mean with padding", v.reshape(-1, 8)[:, 0].mean().round(3),
      " real words only", real_rows[:, 0].mean().round(3))
print("feature 0: batch-norm std with padding", v.reshape(-1, 8)[:, 0].std().round(3),
      " real words only", real_rows[:, 0].std().round(3))

feature 0: batch-norm mean with padding 0.582  real words only 2.054
feature 0: batch-norm std with padding 1.041  real words only 0.895


Pad the same 32 reviews to more and more positions (as if a longer review had joined the batch). Track one real
word: the first word of the first review, feature 0.

In [11]:
rows = []
for T in (longest, 1000, 1500, 2000, 2500):
    v, mask = padded_vectors(T)
    flat = v.reshape(-1, 8)
    bn_out = keras.layers.BatchNormalization()(v, training=True).numpy()
    ln_out = keras.layers.LayerNormalization()(v).numpy()
    rows.append(dict(T=T, padding_share=1 - mask.mean(), bn_mean=flat[:, 0].mean(), bn_std=flat[:, 0].std(),
                     bn_value=bn_out[0, 0, 0], ln_value=ln_out[0, 0, 0],
                     bn_real_only=(v[0, 0, 0] - v[mask][:, 0].mean()) / np.sqrt(v[mask][:, 0].var() + EPS)))
res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
res.to_csv("data/padding_effect.csv", index=False)
pd.DataFrame({"length": lengths}).to_csv("data/imdb_batch_lengths.csv", index=False)

   T  padding_share  bn_mean  bn_std  bn_value  ln_value  bn_real_only
 888         0.7165   0.5822  1.0411    1.7605    1.6284        0.4044
1000         0.7483   0.5170  0.9981    1.9016    1.6284        0.4044
1500         0.8322   0.3447  0.8506    2.4334    1.6284        0.4044
2000         0.8741   0.2585  0.7516    2.8678    1.6284        0.4044
2500         0.8993   0.2068  0.6802    3.2441    1.6284        0.4044


## 4. Training and prediction
Layer norm computes the same thing in both modes; batch norm switches to its moving averages for prediction.

In [12]:
v, _ = padded_vectors(longest)
lnl, bnl = keras.layers.LayerNormalization(), keras.layers.BatchNormalization()
print("layer norm, max |training - prediction|:", np.abs(np.asarray(lnl(v, training=True)) - np.asarray(lnl(v, training=False))).max())
print("batch norm, max |training - prediction|:", np.abs(np.asarray(bnl(v, training=True)) - np.asarray(bnl(v, training=False))).max())

layer norm, max |training - prediction|: 0.0
batch norm, max |training - prediction|: 1.0993371


Parameters for d_model = 512, the size used in the transformer paper.

In [13]:
for layer in (keras.layers.LayerNormalization(), keras.layers.BatchNormalization()):
    m = keras.Sequential([keras.Input(shape=(10, 512)), layer])
    trainable = sum(np.prod(w.shape) for w in m.trainable_weights)
    print(f"{type(layer).__name__:20s} parameters {m.count_params()}, trainable {trainable}")

LayerNormalization   parameters 1024, trainable 1024
BatchNormalization   parameters 2048, trainable 1024
